In [1]:
import xarray as xr
import pandas as pd
import os

In [2]:
# two different files for the motive mission so we first concatenate them

motive1 = xr.open_dataset("sg195_MOTIVE_2024_timeseries.nc")
motive2 = xr.open_dataset("sg195_MOTIVE_2024_timeseries2.nc")

keep_vars = ["time", "depth", "latitude", "longitude", "temperature", "salinity", "dissolved_oxygen_sat", "eng_wlbb2fl_sig470nm", "eng_wlbb2fl_sig700nm", "eng_wlbb2fl_sig695nm", "sg_data_point_dive_number"]

def strip(ds):
    ds = ds[keep_vars]             
    return ds

motive1_ds = strip(motive1)
motive2_ds = strip(motive2)

motive_ds = xr.concat([motive1_ds, motive2_ds], dim="sg_data_point", data_vars="all", coords="minimal", compat="override", join="override")

motive_ds = motive_ds.sortby("ctd_time")

#display(motive_ds)

In [3]:
time = motive_ds['time'][0].values

print(time)

2005-04-06T14:24:48.526000000


In [4]:
#Align time with sg_data_point and apply offset
adjusted_time = pd.to_datetime(motive_ds['time'].values) + pd.DateOffset(years=19, months=7, days=13)

motive_ds = motive_ds.assign_coords(time=('sg_data_point', adjusted_time))

motive_ds['PAR_470nm'] = motive_ds['eng_wlbb2fl_sig470nm']
motive_ds['particle_concentration_700nm'] = motive_ds['eng_wlbb2fl_sig700nm']
motive_ds['chlorophyll_695nm'] = motive_ds['eng_wlbb2fl_sig695nm']
motive_ds['dive_number'] = motive_ds['sg_data_point_dive_number']

# add metadata
motive_ds['PAR_470nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig470nm'
motive_ds['particle_concentration_700nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig700nm'
motive_ds['chlorophyll_695nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig695nm'
motive_ds['dive_number'] .attrs['pre_cleaning_name'] = 'sg_data_point_dive_number'


#Select the relevant variables
new_motive_ds = motive_ds[['time', 'depth',  'latitude','longitude','temperature', 'salinity', 'dissolved_oxygen_sat', 'PAR_470nm', 'particle_concentration_700nm', 'chlorophyll_695nm', 'dive_number']] 
print(new_motive_ds['time'][0].values)

#Convert to DataFrame and save
new_motive_ds.to_dataframe().reset_index().to_csv('sg195_tropics_timeseries_cleaned.csv', index=False)
new_motive_ds.to_netcdf('sg195_tropics_timeseries_cleaned.nc')

2024-11-19T14:24:48.526000000


In [6]:
# two different files for the motive mission so we first concatenate them

motive1_dac = xr.open_dataset("sg195_MOTIVE_2024_timeseries.nc")
motive2_dac = xr.open_dataset("sg195_MOTIVE_2024_timeseries2.nc")

keep_vars = ["depth_avg_curr_east", "depth_avg_curr_north", "start_time", "end_time", "start_latitude", "end_latitude", "start_longitude", "end_longitude"]

motive1_ds_dac = strip(motive1_dac)
motive2_ds_dac = strip(motive2_dac)

motive_ds_dac = xr.concat([motive1_ds_dac, motive2_ds_dac], dim="dive", data_vars="all", coords="minimal", compat="override", join="override")

motive_ds_dac = motive_ds_dac.sortby("start_time")

#display(motive_ds_dac)


In [9]:
#Apply time apply offset (if needed)
adjusted_start = pd.to_datetime(motive_ds_dac['start_time'].values) + pd.DateOffset(years=19, months=7, days=13)
adjusted_end = pd.to_datetime(motive_ds_dac['end_time'].values) + pd.DateOffset(years=19, months=7, days=13)

motive_ds_dac = motive_ds_dac.assign_coords(start_time=('dive', adjusted_start))
motive_ds_dac = motive_ds_dac.assign_coords(end_time=('dive', adjusted_end))

motive_ds_dac['U_DAC'] = motive_ds_dac['depth_avg_curr_east']
motive_ds_dac['V_DAC'] = motive_ds_dac['depth_avg_curr_north']

# add metadata
motive_ds_dac['U_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_east'
motive_ds_dac['V_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_north'

#Select the relevant variables
new_motive_ds = motive_ds_dac[['U_DAC', 'V_DAC', 'start_time', 'end_time', 'start_latitude', 'end_latitude', 'start_longitude', 'end_longitude']]
display(new_motive_ds)

#Convert to DataFrame and save
new_motive_ds.to_dataframe().reset_index().to_csv('sg195_tropics_timeseries_DAC_cleaned.csv', index=False)
new_motive_ds.to_netcdf('sg195_tropics_timeseries_DAC_cleaned.nc')


<xarray.Dataset> Size: 28kB
Dimensions:          (dive: 703)
Coordinates:
    start_time       (dive) datetime64[ns] 6kB 2024-11-19T14:23:46 ... 2025-0...
    end_time         (dive) datetime64[ns] 6kB 2024-11-19T14:48:12 ... 2025-0...
Dimensions without coordinates: dive
Data variables:
    U_DAC            (dive) float32 3kB 0.5582 0.7924 ... -0.07873 -0.08706
    V_DAC            (dive) float32 3kB 0.5208 0.4539 0.3798 ... 0.0843 0.05894
    start_latitude   (dive) float32 3kB 0.7322 0.7448 0.7631 ... 10.04 10.06
    end_latitude     (dive) float32 3kB 0.7386 0.7517 0.7766 ... 10.06 10.07
    start_longitude  (dive) float32 3kB -139.2 -139.2 -139.1 ... -144.3 -144.3
    end_longitude    (dive) float32 3kB -139.2 -139.1 -139.1 ... -144.3 -144.3
Attributes: (12/47)
    project:                         MOTIVE 2024
    title:                           Physical, chemical, and biological data ...
    summary:                         SG195 MOTIVE 2024
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2025-05-14T12:05:07Z
    uuid:                            73016ec2-30bf-11f0-80b3-07c04838153a
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6